In [ ]:
#!pip install -U torchvision

In [1]:
import torch
import torch.nn as nn
import torchvision

In [2]:
def conv_block(inp_filt, out_filt):
    conv = nn.Sequential(nn.Conv2d(inp_filt, out_filt, 3, padding=1),
                         nn.BatchNorm2d(out_filt),
                         nn.ReLU(inplace=True)
                         )
    return conv

In [3]:
!pip install torchinfo
from torchinfo import summary

In [4]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                               nn.ReLU(),
                               nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.avg_pool(x)
        avg_out = self.fc(avg_out)

        max_out = self.max_pool(x)
        max_out = self.fc(max_out)

        out = avg_out + max_out
        out = self.sigmoid(out)
        out = out * x 
        return out

In [5]:
x = torch.ones(1,128, 16, 16)
print(x.shape)

torch.Size([1, 128, 16, 16])


In [6]:
ca = ChannelAttention(128)
summary(ca)

Layer (type:depth-idx)                   Param #
ChannelAttention                         --
├─AdaptiveAvgPool2d: 1-1                 --
├─AdaptiveMaxPool2d: 1-2                 --
├─Sequential: 1-3                        --
│    └─Conv2d: 2-1                       1,024
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       1,024
├─Sigmoid: 1-4                           --
Total params: 2,048
Trainable params: 2,048
Non-trainable params: 0

In [7]:
out = ca(x)

In [8]:
out.shape

torch.Size([1, 128, 16, 16])

In [9]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv1(out)
        out = self.sigmoid(out)
        out = out * x
        return out
        

In [10]:
sa = SpatialAttention()

In [11]:
out = sa(out)
out.shape

torch.Size([1, 128, 16, 16])

In [12]:
def cbam(x):
    in_planes = x.shape[1]
    ca = ChannelAttention(in_planes)
    sa = SpatialAttention(kernel_size=3)
    x = ca(x)
    x = sa(x)
    return x

In [13]:
out = cbam(out)

In [14]:
out.shape

torch.Size([1, 128, 16, 16])

### Bro_RESUNET With CBAM

In [15]:
class conv2blk(nn.Module):
    def __init__(self, inp_filt, out_filt=None):
        super().__init__()
        if not out_filt:
            out_filt = inp_filt            
            
        self.conv = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, out_filt),
        )

    def forward(self, x):
        x = self.conv(x)
        x = cbam(x)
        return x

In [16]:
class decoder_blk(nn.Module):
    def __init__(self, inp_filt):
        super().__init__()
        self.conv = conv2blk(inp_filt)
        self.x2x2cc = nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.x2x2cc(x)
        return x        

In [17]:
backbone = torch.hub.load("pytorch/vision", "resnet50", weights="ResNet50_Weights.IMAGENET1K_V1")

Using cache found in /home/ec2-user/.cache/torch/hub/pytorch_vision_main


In [18]:
class Bro_RESUNET_CBAM(nn.Module):
    def __init__(self, model):
        super().__init__()
        d_f = [2048, 1024, 512, 256, 64]
        self.layer_0 = nn.Sequential(  
            model.conv1, #x/2
            model.bn1,
            model.relu,
            model.maxpool #x/4, 64
        )
        self.layer_1 = model.layer1 #x/4, 256
        self.layer_2 = model.layer2 #x/8, 512
        self.layer_3 = model.layer3 #x/16, 1024
        self.layer_4_bneck = model.layer4 #x/32, 2048

        self.bneck_x_2x = nn.ConvTranspose2d(d_f[0], d_f[1], (2,2), 2) #x/16, 2048, 1024

        self.dec_blk_layer_3 = decoder_blk(d_f[1]) #2048, 1024, 512
        self.dec_blk_layer_2 = decoder_blk(d_f[2]) #1024, 512, 256
        self.dec_blk_layer_1 = conv2blk(d_f[3], d_f[4])  #512, 256, 64
      
        self.final_conv = nn.Sequential(
            nn.ConvTranspose2d(d_f[4]*2, d_f[4], (2,2), 2), #112
            nn.ConvTranspose2d(d_f[4], 1, (2,2), 2), #224 
            #nn.Sigmoid() #we want to use BCELosswithLogits /Cross Entropy so we want to be flexible here
            )

    def forward(self, x):
        s1 = self.layer_0(x)  #224, 56
        s2 = self.layer_1(s1) #56, 56
        s3 = self.layer_2(s2) #56, 28
        s4 = self.layer_3(s3) #28, 14

        out = self.layer_4_bneck(s4) #14, 7
        out = self.bneck_x_2x(out) #7, 14

        out = torch.cat((s4, out), 1) 
        out = self.dec_blk_layer_3(out) #14, 28
        out = torch.cat((s3, out), 1)
        out = self.dec_blk_layer_2(out) #28, 56
        out = torch.cat((s2, out), 1)
        out = self.dec_blk_layer_1(out) #56, 56
        out = torch.cat((s1, out), 1)
        out = self.final_conv(out) #56, 112, 224
        
        return out

In [19]:
model = Bro_RESUNET_CBAM(backbone)

In [20]:
inp = torch.ones(3, 3, 224, 224)
inp.shape

torch.Size([3, 3, 224, 224])

In [21]:
out = model(inp)
out.shape

torch.Size([3, 1, 224, 224])

In [22]:
model

Bro_RESUNET_CBAM(
  (layer_0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer_1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256

In [23]:
summary(model, input_size=(1,3,224,224))

Layer (type:depth-idx)                        Output Shape              Param #
Bro_RESUNET_CBAM                              [1, 1, 224, 224]          --
├─Sequential: 1-1                             [1, 64, 56, 56]           --
│    └─Conv2d: 2-1                            [1, 64, 112, 112]         9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 112, 112]         128
│    └─ReLU: 2-3                              [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-4                         [1, 64, 56, 56]           --
├─Sequential: 1-2                             [1, 256, 56, 56]          --
│    └─Bottleneck: 2-5                        [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                       [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2                  [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                         [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                       [1, 64, 56, 56]           36,864
│    │  